In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
from notebooks.imports import *
from pathlib import Path
from scipy.ndimage import gaussian_filter1d
import pickle
import seaborn as sns

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:
from config import dir_config, ephys_config

compiled_dir = Path(dir_config.data.compiled)
processed_dir = Path(dir_config.data.processed)

In [ ]:
neuron_metadata = pd.read_csv(Path(processed_dir, "neuron_metadata.csv"), index_col=None)

with open(Path(processed_dir, "ephys_neuron_wise.pkl"), "rb") as handle:
    ephys_neuron_wise = pickle.load(handle)


In [ ]:
def filter_trials(trial_info, outcome=1, block_type="equal"):
    trial_info = trial_info[trial_info['reaction_time'].isna() == False]
    trial_info = trial_info[trial_info.task_type == 1]
    trial_info = trial_info[trial_info.outcome == outcome]
    if block_type == "equal":
        trial_info = trial_info[trial_info.prob_toRF == 50]
    else:
        trial_info = trial_info[trial_info.prob_toRF != 50]
    return trial_info

def get_normalized_matrix(baseline_substracted_matrix, peak, trials, trial_numbers):
    
    trial_indices = np.where(np.isin(trial_numbers, trials.trial_number))[0]
    baseline_substracted_matrix = baseline_substracted_matrix[trial_indices, :]
    if peak == 0:
        z_scored_matrix = np.nanmean(baseline_substracted_matrix,axis=0)  # avoid division by zero
    else:
        z_scored_matrix = np.nanmean(baseline_substracted_matrix, axis=0) / peak
    return z_scored_matrix

def get_peak(baseline_substracted_matrix, trials, trial_numbers):
    trial_indices = np.where(np.isin(trial_numbers, trials.trial_number))[0]
    baseline_substracted_matrix = baseline_substracted_matrix[trial_indices, :]

    return np.nanmax(np.abs(np.nanmean(baseline_substracted_matrix,axis=0)))

def get_subsample_matrix(baseline_substracted_matrix, peak, trials, trial_numbers, n_boostraps=100):
    bootstrapped_matrices = []
    for _ in range(n_boostraps):
        bootstrap_trials_idx = np.random.choice(trials.trial_number, size=len(trials), replace=True)
        bootstrap_trials = trials[trials['trial_number'].isin(bootstrap_trials_idx)]
        z_scored_matrix = get_normalized_matrix(baseline_substracted_matrix, peak, bootstrap_trials, trial_numbers)
        bootstrapped_matrices.append(z_scored_matrix)
    return bootstrapped_matrices


In [ ]:
neuron_types = ['visual_phasic', 'visual_tonic', 'visual_motor', 'buildup','motor','unknown']

In [ ]:
visual_start = 0
visual_end = 125

In [ ]:
toRF_visual_matrix, awayRF_visual_matrix = [], []
fig, axs = plt.subplots(2,1, figsize=(6,12))
for neuron_id in neuron_metadata.neuron_id:
    session_id = neuron_metadata.session_id[neuron_metadata.neuron_id == neuron_id].values[0]
    trial_info = pd.read_csv(Path(compiled_dir, session_id, f"{session_id}_trial.csv"), index_col=None)
    trial_info = filter_trials(trial_info)

    baseline_data = ephys_neuron_wise["baseline"][neuron_id]['convolved_spike_trains']
    visual_data = ephys_neuron_wise["visual"][neuron_id]['convolved_spike_trains'][:,visual_start:visual_end]
    baseline_mean = np.nanmean(baseline_data, axis=1)
    baseline_substracted_visual = (visual_data - baseline_mean[:, None])

    trial_numbers = ephys_neuron_wise["visual"][neuron_id]['trial_number']

    toRF_trials = trial_info[trial_info['choice'] == 1]
    awayRF_trials = trial_info[trial_info['choice'] == 0]

    toRF_peak = get_peak(baseline_substracted_visual,toRF_trials,trial_numbers)
    awayRF_peak = get_peak(baseline_substracted_visual,awayRF_trials,trial_numbers)
    peak = max(toRF_peak, awayRF_peak)

    toRF_visual_matrix.append(get_normalized_matrix(baseline_substracted_visual, peak, toRF_trials,trial_numbers))
    awayRF_visual_matrix.append(get_normalized_matrix(baseline_substracted_visual, peak, awayRF_trials,trial_numbers))

# SORT HEATMAP BY PEAK RESPONSE TIME IN EACH NEURON TYPE
toRF_sorted_indices = []
for type in neuron_types:
    type_indices = neuron_metadata.index[neuron_metadata['classification'] == type].tolist()
    sorted_type_indices = np.array(type_indices)[np.argsort([np.nanargmax((toRF_visual_matrix[type_idx])) for type_idx in type_indices])]
    toRF_sorted_indices.append(sorted_type_indices)
# toRF_sorted_indices = np.concatenate(toRF_sorted_indices)

for i in [0,1]:
    start_num = 0
    for type_index, type in enumerate(neuron_types):
        n_neurons = len(toRF_sorted_indices[type_index])
        start_num += n_neurons
        axs[i].axhline(start_num, color='black', linestyle='--', linewidth=1)
        # add text labels for each neuron type
        axs[i].text(visual_end + 5, start_num - n_neurons/2, type.replace('_',' ').title(), va='center', fontsize=10)



# # sort heatmap by peak response time
# toRF_sorted_indices = np.argsort([np.nanargmax((neuron_response)) for neuron_response in toRF_visual_matrix])
# awayRF_sorted_indices = np.argsort([np.nanargmax((neuron_response)) for neuron_response in awayRF_visual_matrix])

sns.heatmap(np.array(toRF_visual_matrix)[np.concatenate(toRF_sorted_indices),:], cmap='coolwarm',ax=axs[0], cbar=False)
sns.heatmap(np.array(awayRF_visual_matrix)[np.concatenate(toRF_sorted_indices),:], cmap='coolwarm',ax=axs[1], cbar=False)
# Single colorbar
cbar_ax = fig.add_axes([1.01, 0.3, 0.02, 0.4])  # [left, bottom, width, height]
sm = plt.cm.ScalarMappable(cmap='coolwarm', norm=plt.Normalize(vmin=-1, vmax=1))
sm.set_array([])
fig.colorbar(sm, cax=cbar_ax)

axs[0].set_title("Visual-aligned population response (Z-scored)");
axs[0].set_xticks([])
axs[1].set_xticks([0, 25, 50, 75, 100], [visual_start, (visual_start+25), (visual_start+50), (visual_start+75), visual_end])
axs[0].set_yticks([])
axs[1].set_yticks([])
axs[0].set_ylabel("Peak-Latency Sorted Neuron \n(To RF)")
axs[1].set_ylabel("Peak-Latency Sorted Neuron \n(Away RF)")
axs[1].set_xlabel("Time (ms) relative to target onset");
plt.tight_layout()

#### boostrap

In [ ]:

# Storage dictionaries
toRF_bootstrap_visual_dict = {}
awayRF_bootstrap_visual_dict = {}

# Pre-load all trial CSVs by session
session_cache = {}

for idx_neuron, neuron_id in enumerate(neuron_metadata.neuron_id):

    session_id = neuron_metadata.session_id[
        neuron_metadata.neuron_id == neuron_id
    ].values[0]

    # Load session trials once
    if session_id not in session_cache:
        df = pd.read_csv(Path(compiled_dir, session_id, f"{session_id}_trial.csv"))
        session_cache[session_id] = filter_trials(df)
    trial_info = session_cache[session_id]


    # Extract baseline + visual data once
    baseline_data = ephys_neuron_wise["baseline"][neuron_id]['convolved_spike_trains']
    visual_data = ephys_neuron_wise["visual"][neuron_id]['convolved_spike_trains'][:, visual_start:visual_end]
    trial_numbers = ephys_neuron_wise["visual"][neuron_id]['trial_number']

    baseline_mean = np.nanmean(baseline_data, axis=1)
    baseline_sub_visual = visual_data - baseline_mean[:, None]

    # Split trials
    toRF_trials = trial_info[trial_info['choice'] == 1]
    awayRF_trials = trial_info[trial_info['choice'] == 0]

    # Compute peak
    toRF_peak = get_peak(baseline_sub_visual, toRF_trials, trial_numbers)
    awayRF_peak = get_peak(baseline_sub_visual, awayRF_trials, trial_numbers)
    peak = max(toRF_peak, awayRF_peak)

    # Bootstrap matrices
    toRF_boot = get_subsample_matrix(baseline_sub_visual, peak, toRF_trials, trial_numbers)
    awayRF_boot = get_subsample_matrix(baseline_sub_visual, peak, awayRF_trials, trial_numbers)

    # Save
    toRF_bootstrap_visual_dict[neuron_id] = toRF_boot
    awayRF_bootstrap_visual_dict[neuron_id] = awayRF_boot



In [ ]:
# Precompute figure index mapping:
flat_sorted = np.concatenate(toRF_sorted_indices)
neuron_to_figidx = {neuron_idx: np.where(flat_sorted == neuron_idx)[0][0]
                    for neuron_idx in range(len(neuron_metadata))}

# Plot setup
n_neurons = len(neuron_metadata.neuron_id)
fig_to, axs_to = plt.subplots(n_neurons, 1, figsize=(3, n_neurons/8))
fig_away, axs_away = plt.subplots(n_neurons, 1, figsize=(3, n_neurons/8))
# Compute boundaries between types
type_sizes = [len(arr) for arr in toRF_sorted_indices]
type_boundaries = np.cumsum(type_sizes)
for i, size in enumerate(type_sizes):
    # Row index of block center
    if i == 0:
        start = 0
    else:
        start = type_boundaries[i-1]
    end = type_boundaries[i]
    center = (start + end) // 2

    # ---- Label on the right side ----
    axs_to[center].text(
        1.02, 0.5, neuron_types[i],
        transform=axs_to[center].transAxes,
        va='center', ha='left', fontsize=10
    )
    axs_away[center].text(
        1.02, 0.5, neuron_types[i],
        transform=axs_away[center].transAxes,
        va='center', ha='left', fontsize=10
    )

for idx_neuron, neuron_id in enumerate(neuron_metadata.neuron_id):
    # Figure index
    fig_idx = neuron_to_figidx[idx_neuron]
    # Plotting
    for row in toRF_bootstrap_visual_dict[neuron_id]:
        axs_to[fig_idx].plot(row, color='lightblue', alpha=0.3, linewidth=1)
    axs_to[fig_idx].plot(toRF_visual_matrix[idx_neuron], color='blue', linewidth=1)
    for row in awayRF_bootstrap_visual_dict[neuron_id]:
        axs_away[fig_idx].plot(row, color='lightblue', alpha=0.3, linewidth=1)
    axs_away[fig_idx].plot(awayRF_visual_matrix[idx_neuron], color='blue', linewidth=1)

    # formatting
    if fig_idx < n_neurons - 1:
        axs_to[fig_idx].set_xticks([])
        axs_away[fig_idx].set_xticks([])
    else:
        axs_to[fig_idx].set_xlabel("Time (ms) relative to target onset")
        axs_away[fig_idx].set_xlabel("Time (ms) relative to target onset")
        axs_to[fig_idx].set_xticks([0,25,50,75,100], 
                                   [visual_start, visual_start+25, visual_start+50, visual_start+75, visual_end])
        axs_away[fig_idx].set_xticks([0,25,50,75,100], 
                                     [visual_start, visual_start+25, visual_start+50, visual_start+75, visual_end])

    for ax in [axs_to[fig_idx], axs_away[fig_idx]]:
        ax.set_yticks([])
        # ax.set_ylim([-1,1])
        for spine in ax.spines.values():
            spine.set_visible(False)


# Add dashed separators between cell types
for boundary in type_boundaries[:-1]:  # omit final line
    # Add dashed line in both panels
    axs_to[boundary].axhline(-0.5, color='black', linestyle='--', linewidth=1)
    axs_away[boundary].axhline(-0.5, color='black', linestyle='--', linewidth=1)

fig_to.tight_layout()
fig_away.tight_layout()

#### coherence-separated

In [ ]:
coh_levels = [0,6,20,50]
toRF_visual_matrix, awayRF_visual_matrix = {coh:[] for coh in coh_levels}, {coh:[] for coh in coh_levels}
for neuron_id in neuron_metadata.neuron_id:
    session_id = neuron_metadata.session_id[neuron_metadata.neuron_id == neuron_id].values[0]
    trial_info = pd.read_csv(Path(compiled_dir, session_id, f"{session_id}_trial.csv"), index_col=None)
    trial_info = filter_trials(trial_info)

    baseline_data = ephys_neuron_wise["baseline"][neuron_id]['convolved_spike_trains']
    visual_data = ephys_neuron_wise["visual"][neuron_id]['convolved_spike_trains'][:,visual_start:visual_end]
    baseline_mean = np.nanmean(baseline_data, axis=1)
    baseline_substracted_visual = (visual_data - baseline_mean[:, None])

    trial_numbers = ephys_neuron_wise["visual"][neuron_id]['trial_number']

    toRF_trials = trial_info[trial_info['choice'] == 1]
    awayRF_trials = trial_info[trial_info['choice'] == 0]

    toRF_peak = []
    awayRF_peak = []
    for coh in coh_levels:
        coh_toRF_trials = toRF_trials[toRF_trials['coherence'] == coh]
        coh_awayRF_trials = awayRF_trials[awayRF_trials['coherence'] == coh]
        toRF_peak.append(get_peak(baseline_substracted_visual, coh_toRF_trials, trial_numbers))
        awayRF_peak.append(get_peak(baseline_substracted_visual, coh_awayRF_trials, trial_numbers))
    peak = np.max([toRF_peak, awayRF_peak])

    for coh in coh_levels:
        coh_toRF_trials = toRF_trials[toRF_trials['coherence'] == coh]
        coh_awayRF_trials = awayRF_trials[awayRF_trials['coherence'] == coh]
        
        toRF_cue_neuron = get_normalized_matrix(baseline_substracted_visual, peak, coh_toRF_trials, trial_numbers)
        awayRF_cue_neuron = get_normalized_matrix(baseline_substracted_visual, peak, coh_awayRF_trials, trial_numbers)

        toRF_visual_matrix[coh].append(toRF_cue_neuron)
        awayRF_visual_matrix[coh].append(awayRF_cue_neuron)


fig, axs = plt.subplots(2, len(coh_levels), figsize=(6*len(coh_levels), 12))


# toRF_sorted_indices = np.argsort([np.nanargmax(neuron_response) for neuron_response in toRF_visual_matrix[50]])
# awayRF_sorted_indices = np.argsort([np.nanargmax(neuron_response) for neuron_response in awayRF_visual_matrix[0]])

for coh_index, coh in enumerate(coh_levels):
    # sort indices
    # toRF_sorted_indices = np.argsort([np.nanargmax(neuron_response) for neuron_response in toRF_visual_matrix[coh]])
    # awayRF_sorted_indices = np.argsort([np.nanargmax(neuron_response) for neuron_response in awayRF_visual_matrix[coh]])
    toRF_sorted_indices = []
    awayRF_sorted_indices = []
    for type in neuron_types:
        type_indices = neuron_metadata.index[neuron_metadata['classification'] == type].tolist()
        sorted_type_indices_toRF = np.array(type_indices)[np.argsort([np.nanargmax((toRF_visual_matrix[coh][type_idx])) for type_idx in type_indices])]
        toRF_sorted_indices.append(sorted_type_indices_toRF)
        sorted_type_indices_awayRF = np.array(type_indices)[np.argsort([np.nanargmax((awayRF_visual_matrix[coh][type_idx])) for type_idx in type_indices])]
        awayRF_sorted_indices.append(sorted_type_indices_awayRF)

    # plot heatmaps
    sns.heatmap(np.array(toRF_visual_matrix[coh])[np.concatenate(toRF_sorted_indices)],
                cmap='coolwarm', ax=axs[0, coh_index], vmin=-1, vmax=1, cbar=False)
    sns.heatmap(np.array(awayRF_visual_matrix[coh])[np.concatenate(toRF_sorted_indices)],
                cmap='coolwarm', ax=axs[1, coh_index], vmin=-1, vmax=1, cbar=False)

    # styling
    axs[0, coh_index].set_title(f"toRF ({coh}%)")
    axs[1, coh_index].set_title(f"awayRF ({coh}%)")

    for row in [0, 1]:
        axs[row, coh_index].set_yticks([])
        axs[row, coh_index].axvline(0, color='red', linestyle='--')
        axs[row, coh_index].set_ylabel("Peak-Latency Sorted Neuron", fontsize=15)

    axs[0, coh_index].set_xticks([])
    axs[1, coh_index].set_xticks(np.arange(0, 100, 25))
    axs[1, coh_index].set_xlabel("Time (ms) relative to target onset")

# SORT HEATMAP BY PEAK RESPONSE TIME IN EACH NEURON TYPE
for i in [0,1]:
    start_num = 0
    for type_index, type in enumerate(neuron_types):
        n_neurons = len(toRF_sorted_indices[type_index])
        start_num += n_neurons
        for coh_index, coh in enumerate(coh_levels):
            axs[i,coh_index].axhline(start_num, color='black', linestyle='--', linewidth=1)
        # add text labels for each neuron type
        axs[i,3].text(visual_end + 5, start_num - n_neurons/2, type.replace('_',' ').title(), va='center', fontsize=10)


# Single colorbar
cbar_ax = fig.add_axes([1.01, 0.2, 0.01, 0.6])  # [left, bottom, width, height]
sm = plt.cm.ScalarMappable(cmap='coolwarm', norm=plt.Normalize(vmin=-1, vmax=1))
sm.set_array([])
fig.colorbar(sm, cax=cbar_ax)

plt.suptitle("Target-aligned population response (Z-scored)", fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


#### Cue

In [ ]:
toRF_cue_neuron.shape

In [ ]:
cue_end = 400
toRF_cue_matrix, awayRF_cue_matrix = [], []
fig, axs = plt.subplots(2,1, figsize=(6,12))
for neuron_id in neuron_metadata.neuron_id:
    session_id = neuron_metadata.session_id[neuron_metadata.neuron_id == neuron_id].values[0]
    trial_info = pd.read_csv(Path(compiled_dir, session_id, f"{session_id}_trial.csv"), index_col=None)
    trial_info = filter_trials(trial_info)

    baseline_data = ephys_neuron_wise["baseline"][neuron_id]['convolved_spike_trains']
    cue_data = ephys_neuron_wise["cue"][neuron_id]['convolved_spike_trains']#[:,0:cue_end]
    baseline_mean = np.nanmean(baseline_data, axis=1)
    baseline_substracted_cue = (cue_data - baseline_mean[:, None])

    trial_numbers = ephys_neuron_wise["cue"][neuron_id]['trial_number']

    toRF_trials = trial_info[trial_info['choice'] == 1]
    awayRF_trials = trial_info[trial_info['choice'] == 0]

    toRF_peak = get_peak(baseline_substracted_cue,toRF_trials,trial_numbers)
    awayRF_peak = get_peak(baseline_substracted_cue,awayRF_trials,trial_numbers)
    peak = max(toRF_peak, awayRF_peak)

    toRF_cue_neuron = get_normalized_matrix(baseline_substracted_cue, peak, toRF_trials,trial_numbers)
    awayRF_cue_neuron = get_normalized_matrix(baseline_substracted_cue, peak, awayRF_trials,trial_numbers)
    # toRF_reaction_times = np.nanmin(toRF_trials['reaction_time'])
    # if toRF_reaction_times + 50 < len(toRF_cue_neuron):
    #     toRF_cue_neuron[int(toRF_reaction_times + 50):] = np.nan
    # awayRF_reaction_times = np.nanmin(awayRF_trials['reaction_time'])
    # if awayRF_reaction_times + 50 < len(awayRF_cue_neuron):
    #     awayRF_cue_neuron[int(awayRF_reaction_times + 50):] = np.nan

    toRF_cue_matrix.append(toRF_cue_neuron)
    awayRF_cue_matrix.append(awayRF_cue_neuron)

# SORT HEATMAP BY PEAK RESPONSE TIME IN EACH NEURON TYPE
toRF_sorted_indices = []
for type in neuron_types:
    type_indices = neuron_metadata.index[neuron_metadata['classification'] == type].tolist()
    sorted_type_indices = np.array(type_indices)[np.argsort([np.nanargmax((toRF_cue_matrix[type_idx])) for type_idx in type_indices])]
    toRF_sorted_indices.append(sorted_type_indices)
# toRF_sorted_indices = np.concatenate(toRF_sorted_indices)

for i in [0,1]:
    start_num = 0
    for type_index, type in enumerate(neuron_types):
        n_neurons = len(toRF_sorted_indices[type_index])
        start_num += n_neurons
        axs[i].axhline(start_num, color='black', linestyle='--', linewidth=1)
        # add text labels for each neuron type
        axs[i].text(len(toRF_cue_neuron)+5, start_num - n_neurons/2, type.replace('_',' ').title(), va='center', fontsize=10)

# # sort heatmap by peak response time
# toRF_sorted_indices = np.argsort([np.nanargmax((neuron_response)) for neuron_response in toRF_cue_matrix])
# awayRF_sorted_indices = np.argsort([np.nanargmax((neuron_response)) for neuron_response in awayRF_cue_matrix])

sns.heatmap(np.array(toRF_cue_matrix)[np.concatenate(toRF_sorted_indices),:], cmap='coolwarm',ax=axs[0], cbar=False)
sns.heatmap(np.array(awayRF_cue_matrix)[np.concatenate(toRF_sorted_indices),:], cmap='coolwarm',ax=axs[1], cbar=False)
# Single colorbar
cbar_ax = fig.add_axes([1.01, 0.3, 0.02, 0.4])  # [left, bottom, width, height]
sm = plt.cm.ScalarMappable(cmap='coolwarm', norm=plt.Normalize(vmin=-1, vmax=1))
sm.set_array([])
fig.colorbar(sm, cax=cbar_ax)

axs[0].set_title("Cue-aligned population response (Z-scored)");
axs[0].set_xticks([])
axs[1].set_xticks(np.arange(0, 900, 100), np.arange(-100, 800, 100))
axs[0].set_yticks([])
axs[1].set_yticks([])
axs[0].set_ylabel("Peak-Latency Sorted Neuron \n(To RF)")
axs[1].set_ylabel("Peak-Latency Sorted Neuron \n(Away RF)")
axs[1].set_xlabel("Time (ms) relative to cue onset");
plt.tight_layout()

In [ ]:
coh_levels = [0,6,20,50]

toRF_cue_matrix = {coh:[] for coh in coh_levels}
awayRF_cue_matrix = {coh:[] for coh in coh_levels}

for neuron_id in neuron_metadata.neuron_id:
    
    session_id = neuron_metadata.session_id[neuron_metadata.neuron_id == neuron_id].values[0]
    trial_info = pd.read_csv(Path(compiled_dir, session_id, f"{session_id}_trial.csv"), index_col=None)
    trial_info = filter_trials(trial_info)
    
    baseline_data = ephys_neuron_wise["baseline"][neuron_id]['convolved_spike_trains']
    baseline_mean = np.nanmean(baseline_data, axis=1)

    cue_data = ephys_neuron_wise["cue"][neuron_id]['convolved_spike_trains']#[:,0:cue_end]
    baseline_substracted_cue = (cue_data - baseline_mean[:, None])

    trial_numbers = ephys_neuron_wise["cue"][neuron_id]['trial_number']

    toRF_peak = []
    awayRF_peak = []
    toRF_trials = trial_info[trial_info['choice'] == 1]
    awayRF_trials = trial_info[trial_info['choice'] == 0]
    for coh in coh_levels:
        coh_toRF_trials = toRF_trials[toRF_trials['coherence'] == coh]
        coh_awayRF_trials = awayRF_trials[awayRF_trials['coherence'] == coh]
        toRF_peak.append(get_peak(baseline_substracted_cue, coh_toRF_trials, trial_numbers))
        awayRF_peak.append(get_peak(baseline_substracted_cue, coh_awayRF_trials, trial_numbers))
    peak = np.max([toRF_peak, awayRF_peak])

    for coh in coh_levels:
        coh_toRF_trials = toRF_trials[toRF_trials['coherence'] == coh]
        coh_awayRF_trials = awayRF_trials[awayRF_trials['coherence'] == coh]
        
        toRF_cue_neuron = get_normalized_matrix(baseline_substracted_cue, peak, coh_toRF_trials, trial_numbers)
        awayRF_cue_neuron = get_normalized_matrix(baseline_substracted_cue, peak, coh_awayRF_trials, trial_numbers)
    
        # toRF_reaction_times = np.nanmin(coh_toRF_trials['reaction_time'])
        # if toRF_reaction_times + 50 < len(toRF_cue_neuron):
        #     toRF_cue_neuron[int(toRF_reaction_times + 50):] = np.nan
        toRF_cue_matrix[coh].append(toRF_cue_neuron)
        # awayRF_reaction_times = np.nanmin(coh_awayRF_trials['reaction_time'])
        # if awayRF_reaction_times + 50 < len(awayRF_cue_neuron):
        #     awayRF_cue_neuron[int(awayRF_reaction_times + 50):] = np.nan
        awayRF_cue_matrix[coh].append(awayRF_cue_neuron)

In [ ]:
fig, axs = plt.subplots(2, len(coh_levels), figsize=(6*len(coh_levels), 12))

# toRF_sorted_indices = np.argsort([np.nanargmax(neuron_response) for neuron_response in toRF_cue_matrix[0]])
# awayRF_sorted_indices = np.argsort([np.nanargmax(neuron_response) for neuron_response in awayRF_cue_matrix[0]])
for coh_index, coh in enumerate(coh_levels):
    # sort indices
    # toRF_sorted_indices = np.argsort([np.nanargmax(neuron_response) for neuron_response in toRF_cue_matrix[coh]])
    # awayRF_sorted_indices = np.argsort([np.nanargmax(neuron_response) for neuron_response in awayRF_cue_matrix[coh]])

    toRF_sorted_indices = []
    awayRF_sorted_indices = []
    for type in neuron_types:
        type_indices = neuron_metadata.index[neuron_metadata['classification'] == type].tolist()
        sorted_type_indices_toRF = np.array(type_indices)[np.argsort([np.nanargmax((toRF_cue_matrix[50][type_idx])) for type_idx in type_indices])]
        toRF_sorted_indices.append(sorted_type_indices_toRF)
        sorted_type_indices_awayRF = np.array(type_indices)[np.argsort([np.nanargmax((awayRF_cue_matrix[50][type_idx])) for type_idx in type_indices])]
        awayRF_sorted_indices.append(sorted_type_indices_awayRF)

    # plot heatmaps
    sns.heatmap(np.array(toRF_cue_matrix[coh])[np.concatenate(toRF_sorted_indices)],
                cmap='coolwarm', ax=axs[0, coh_index], vmin=-1, vmax=1, cbar=False)
    sns.heatmap(np.array(awayRF_cue_matrix[coh])[np.concatenate(awayRF_sorted_indices)],
                cmap='coolwarm', ax=axs[1, coh_index], vmin=-1, vmax=1, cbar=False)
    # styling
    axs[0, coh_index].set_title(f"toRF ({coh}%)")
    axs[1, coh_index].set_title(f"awayRF ({coh}%)")

    for row in [0, 1]:
        axs[row, coh_index].set_yticks([])
        axs[row, coh_index].axvline(100, color='red', linestyle='--')
        axs[row, coh_index].set_ylabel("Peak-Latency Sorted Neuron", fontsize=15)

    axs[0, coh_index].set_xticks([])
    axs[1, coh_index].set_xticks(np.arange(0, 900, 100))
    axs[1, coh_index].set_xticklabels(np.arange(-100, 800, 100))
    axs[1, coh_index].set_xlabel("Time (ms) relative to cue onset")

for i in [0,1]:
    start_num = 0
    for type_index, type in enumerate(neuron_types):
        n_neurons = len(toRF_sorted_indices[type_index])
        start_num += n_neurons
        for coh_index, coh in enumerate(coh_levels):
            axs[i, coh_index].axhline(start_num, color='black', linestyle='--', linewidth=1)
        # add text labels for each neuron type
        axs[i, 3].text(len(toRF_cue_neuron)+5, start_num - n_neurons/2, type.replace('_',' ').title(), va='center', fontsize=10)

# Single colorbar
cbar_ax = fig.add_axes([1.01, 0.2, 0.01, 0.6])  # [left, bottom, width, height]
sm = plt.cm.ScalarMappable(cmap='coolwarm', norm=plt.Normalize(vmin=-1, vmax=1))
sm.set_array([])
fig.colorbar(sm, cax=cbar_ax)

plt.suptitle("Cue-aligned population response (Z-scored)", fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


## Saccade

In [ ]:
saccade_start = 0
saccade_end = -40

In [ ]:

toRF_saccade_matrix, awayRF_saccade_matrix = [], []
fig, axs = plt.subplots(2, 1, figsize=(6, 12))

for neuron_id in neuron_metadata.neuron_id:
    session_id = neuron_metadata.session_id[neuron_metadata.neuron_id == neuron_id].values[0]
    trial_info = pd.read_csv(Path(compiled_dir, session_id, f"{session_id}_trial.csv"), index_col=None)
    trial_info = filter_trials(trial_info)

    baseline_data = ephys_neuron_wise["baseline"][neuron_id]['convolved_spike_trains']
    saccade_data = ephys_neuron_wise["response"][neuron_id]['convolved_spike_trains'][:, saccade_start:saccade_end]

    baseline_mean = np.nanmean(baseline_data, axis=1)
    baseline_substracted_saccade = saccade_data - baseline_mean[:, None]

    trial_numbers = ephys_neuron_wise["response"][neuron_id]['trial_number']

    toRF_trials = trial_info[trial_info['choice'] == 1]
    awayRF_trials = trial_info[trial_info['choice'] == 0]

    toRF_peak = get_peak(baseline_substracted_saccade, toRF_trials, trial_numbers)
    awayRF_peak = get_peak(baseline_substracted_saccade, awayRF_trials, trial_numbers)
    peak = max(toRF_peak, awayRF_peak)


    # toRF_reaction_times = np.nanmin(toRF_trials['reaction_time'])
    toRF_saccade_neuron = get_normalized_matrix(baseline_substracted_saccade, peak, toRF_trials, trial_numbers)
    # if toRF_reaction_times  < len(toRF_saccade_neuron):
    #     toRF_saccade_neuron[:len(toRF_saccade_neuron)-int(toRF_reaction_times)] = np.nan
    toRF_saccade_matrix.append(toRF_saccade_neuron)

    # awayRF_reaction_times = np.nanmin(awayRF_trials['reaction_time'])
    awayRF_saccade_neuron = get_normalized_matrix(baseline_substracted_saccade, peak, awayRF_trials, trial_numbers)
    # if awayRF_reaction_times  < len(awayRF_saccade_neuron):
    #     awayRF_saccade_neuron[:len(awayRF_saccade_neuron)-int(awayRF_reaction_times)] = np.nan
    awayRF_saccade_matrix.append(awayRF_saccade_neuron)

# Convert to arrays
toRF_saccade_matrix = np.array(toRF_saccade_matrix)
awayRF_saccade_matrix = np.array(awayRF_saccade_matrix)

# SORT HEATMAP BY PEAK RESPONSE TIME IN EACH NEURON TYPE
toRF_sorted_indices = []
for type in neuron_types:
    type_indices = neuron_metadata.index[neuron_metadata['classification'] == type].tolist()
    sorted_type_indices = np.array(type_indices)[np.argsort([np.nanargmax((toRF_saccade_matrix[type_idx])) for type_idx in type_indices])][::-1]
    toRF_sorted_indices.append(sorted_type_indices)
# toRF_sorted_indices = np.concatenate(toRF_sorted_indices)

for i in [0,1]:
    start_num = 0
    for type_index, type in enumerate(neuron_types):
        n_neurons = len(toRF_sorted_indices[type_index])
        start_num += n_neurons
        axs[i].axhline(start_num, color='black', linestyle='--', linewidth=1)
        # add text labels for each neuron type
        axs[i].text(toRF_saccade_matrix.shape[1]+5, start_num - n_neurons/2, type.replace('_',' ').title(), va='center', fontsize=10)

# # Sort heatmaps by peak response
# toRF_sorted_indices = np.argsort([np.nanargmax(neuron_response) for neuron_response in toRF_saccade_matrix])[::-1]
# awayRF_sorted_indices = np.argsort([np.nanargmax(neuron_response) for neuron_response in awayRF_saccade_matrix])[::-1]

sns.heatmap(toRF_saccade_matrix[np.concatenate(toRF_sorted_indices), :], cmap='coolwarm', ax=axs[0], vmin=-1, vmax=1, cbar=False)
sns.heatmap(awayRF_saccade_matrix[np.concatenate(toRF_sorted_indices), :], cmap='coolwarm', ax=axs[1], vmin=-1, vmax=1, cbar=False)

axs[0].set_title("Saccade-aligned population response (Z-scored)")
axs[0].set_xticks([])
axs[1].set_xticks([0, 100, 200, 300, 310])
axs[1].set_xticklabels([-300, -200, -100, 0, 10])
axs[0].axvline(300, color='red', linestyle='--')
axs[1].axvline(300, color='red', linestyle='--')
axs[0].set_yticks([])
axs[1].set_yticks([])
axs[0].set_ylabel("Peak-Latency Sorted Neuron \n(To RF)")
axs[1].set_ylabel("Peak-Latency Sorted Neuron \n(Away RF)")
axs[1].set_xlabel("Time (ms) relative to saccade onset")

# Single colorbar
cbar_ax = fig.add_axes([0.995, 0.3, 0.02, 0.4])  # [left, bottom, width, height]
sm = plt.cm.ScalarMappable(cmap='coolwarm', norm=plt.Normalize(vmin=-1, vmax=1))
sm.set_array([])
fig.colorbar(sm, cax=cbar_ax)


plt.tight_layout()

#### boostrap

In [ ]:

# Storage dictionaries
toRF_bootstrap_saccade_dict = {}
awayRF_bootstrap_saccade_dict = {}

# Pre-load all trial CSVs by session
session_cache = {}

for idx_neuron, neuron_id in enumerate(neuron_metadata.neuron_id):

    session_id = neuron_metadata.session_id[
        neuron_metadata.neuron_id == neuron_id
    ].values[0]

    # Load session trials once
    if session_id not in session_cache:
        df = pd.read_csv(Path(compiled_dir, session_id, f"{session_id}_trial.csv"))
        session_cache[session_id] = filter_trials(df)
    trial_info = session_cache[session_id]


    # Extract baseline + visual data once
    baseline_data = ephys_neuron_wise["baseline"][neuron_id]['convolved_spike_trains']
    saccade_data = ephys_neuron_wise["response"][neuron_id]['convolved_spike_trains'][:, saccade_start:saccade_end]
    trial_numbers = ephys_neuron_wise["visual"][neuron_id]['trial_number']

    baseline_mean = np.nanmean(baseline_data, axis=1)
    baseline_sub_saccade = saccade_data - baseline_mean[:, None]

    # Split trials
    toRF_trials = trial_info[trial_info['choice'] == 1]
    awayRF_trials = trial_info[trial_info['choice'] == 0]

    # Compute peak
    toRF_peak = get_peak(baseline_sub_saccade, toRF_trials, trial_numbers)
    awayRF_peak = get_peak(baseline_sub_saccade, awayRF_trials, trial_numbers)
    peak = max(toRF_peak, awayRF_peak)

    # Bootstrap matrices
    toRF_boot = get_subsample_matrix(baseline_sub_saccade, peak, toRF_trials, trial_numbers)
    awayRF_boot = get_subsample_matrix(baseline_sub_saccade, peak, awayRF_trials, trial_numbers)

    # Save
    toRF_bootstrap_saccade_dict[neuron_id] = toRF_boot
    awayRF_bootstrap_saccade_dict[neuron_id] = awayRF_boot



In [ ]:
# Precompute figure index mapping:
flat_sorted = np.concatenate(toRF_sorted_indices)
neuron_to_figidx = {neuron_idx: np.where(flat_sorted == neuron_idx)[0][0]
                    for neuron_idx in range(len(neuron_metadata))}

# Plot setup
n_neurons = len(neuron_metadata.neuron_id)
fig_to, axs_to = plt.subplots(n_neurons, 1, figsize=(2, n_neurons/8))
fig_away, axs_away = plt.subplots(n_neurons, 1, figsize=(2, n_neurons/8))
# Compute boundaries between types
type_sizes = [len(arr) for arr in toRF_sorted_indices]
type_boundaries = np.cumsum(type_sizes)
for i, size in enumerate(type_sizes):
    # Row index of block center
    if i == 0:
        start = 0
    else:
        start = type_boundaries[i-1]
    end = type_boundaries[i]
    center = (start + end) // 2

    # ---- Label on the right side ----
    axs_to[center].text(
        1.02, 0.5, neuron_types[i],
        transform=axs_to[center].transAxes,
        va='center', ha='left', fontsize=10
    )
    axs_away[center].text(
        1.02, 0.5, neuron_types[i],
        transform=axs_away[center].transAxes,
        va='center', ha='left', fontsize=10
    )

for idx_neuron, neuron_id in enumerate(neuron_metadata.neuron_id):
    # Figure index
    fig_idx = neuron_to_figidx[idx_neuron]
    # Plotting
    for row in toRF_bootstrap_saccade_dict[neuron_id]:
        axs_to[fig_idx].plot(row, color='lightblue', alpha=0.3, linewidth=1)
    axs_to[fig_idx].plot(toRF_saccade_matrix[idx_neuron], color='blue', linewidth=1)
    for row in awayRF_bootstrap_saccade_dict[neuron_id]:
        axs_away[fig_idx].plot(row, color='lightblue', alpha=0.3, linewidth=1)
    axs_away[fig_idx].plot(awayRF_saccade_matrix[idx_neuron], color='blue', linewidth=1)

    # formatting
    if fig_idx < n_neurons - 1:
        axs_to[fig_idx].set_xticks([])
        axs_away[fig_idx].set_xticks([])
    else:
        axs_to[fig_idx].set_xlabel("Time (ms) relative to saccade onset")
        axs_away[fig_idx].set_xlabel("Time (ms) relative to saccade onset")
        axs_to[fig_idx].set_xticks([0, 100, 200, 300, 310], [-300, -200, -100, 0, 10])
        axs_away[fig_idx].set_xticks([0, 100, 200, 300, 310], [-300, -200, -100, 0, 10])

    for ax in [axs_to[fig_idx], axs_away[fig_idx]]:
        ax.set_yticks([])
        ax.set_ylim([-1,1])
        for spine in ax.spines.values():
            spine.set_visible(False)


# Add dashed separators between cell types
for boundary in type_boundaries[:-1]:  # omit final line
    # Add dashed line in both panels
    axs_to[boundary].axhline(-0.5, color='black', linestyle='--', linewidth=1)
    axs_away[boundary].axhline(-0.5, color='black', linestyle='--', linewidth=1)

fig_to.tight_layout()
fig_away.tight_layout()

#### coherence separated

In [ ]:
coh_levels = [0,6,20,50]
toRF_saccade_matrix, awayRF_saccade_matrix = {coh:[] for coh in coh_levels}, {coh:[] for coh in coh_levels}
for neuron_id in neuron_metadata.neuron_id:
    session_id = neuron_metadata.session_id[neuron_metadata.neuron_id == neuron_id].values[0]
    trial_info = pd.read_csv(Path(compiled_dir, session_id, f"{session_id}_trial.csv"), index_col=None)
    trial_info = filter_trials(trial_info)

    baseline_data = ephys_neuron_wise["baseline"][neuron_id]['convolved_spike_trains']
    saccade_data = ephys_neuron_wise["response"][neuron_id]['convolved_spike_trains'][:, saccade_start:saccade_end]
    baseline_mean = np.nanmean(baseline_data, axis=1)
    baseline_substracted_saccade = (saccade_data - baseline_mean[:, None])

    trial_numbers = ephys_neuron_wise["response"][neuron_id]['trial_number']

    toRF_trials = trial_info[trial_info['choice'] == 1]
    awayRF_trials = trial_info[trial_info['choice'] == 0]

    toRF_peak = []
    awayRF_peak = []
    for coh in coh_levels:
        coh_toRF_trials = toRF_trials[toRF_trials['coherence'] == coh]
        coh_awayRF_trials = awayRF_trials[awayRF_trials['coherence'] == coh]
        toRF_peak.append(get_peak(baseline_substracted_saccade, coh_toRF_trials, trial_numbers))
        awayRF_peak.append(get_peak(baseline_substracted_saccade, coh_awayRF_trials, trial_numbers))
    peak = np.max([toRF_peak, awayRF_peak])

    for coh in coh_levels:
        coh_toRF_trials = toRF_trials[toRF_trials['coherence'] == coh]
        coh_awayRF_trials = awayRF_trials[awayRF_trials['coherence'] == coh]

        toRF_saccade_neuron = get_normalized_matrix(baseline_substracted_saccade, peak, coh_toRF_trials, trial_numbers)
        awayRF_saccade_neuron = get_normalized_matrix(baseline_substracted_saccade, peak, coh_awayRF_trials, trial_numbers)

        # toRF_reaction_times = np.nanmin(coh_toRF_trials['reaction_time'])
        # if toRF_reaction_times < len(toRF_saccade_neuron):
        #     toRF_saccade_neuron[:len(toRF_saccade_neuron) - int(toRF_reaction_times)] = np.nan

        # awayRF_reaction_times = np.nanmin(coh_awayRF_trials['reaction_time'])
        # if awayRF_reaction_times < len(awayRF_saccade_neuron):
        #     awayRF_saccade_neuron[:len(awayRF_saccade_neuron) - int(awayRF_reaction_times)] = np.nan

        toRF_saccade_matrix[coh].append(toRF_saccade_neuron)
        awayRF_saccade_matrix[coh].append(awayRF_saccade_neuron)


fig, axs = plt.subplots(2, len(coh_levels), figsize=(6*len(coh_levels), 12))

for coh_index, coh in enumerate(coh_levels):
    # sort indices
    # toRF_sorted_indices = np.argsort([np.nanargmax(neuron_response) for neuron_response in toRF_saccade_matrix[coh]])[::-1]
    # awayRF_sorted_indices = np.argsort([np.nanargmax(neuron_response) for neuron_response in awayRF_saccade_matrix[coh]])[::-1]

    # SORT HEATMAP BY PEAK RESPONSE TIME IN EACH NEURON TYPE
    toRF_sorted_indices = []
    awayRF_sorted_indices = []
    for type in neuron_types:
        type_indices = neuron_metadata.index[neuron_metadata['classification'] == type].tolist()
        sorted_type_indices_toRF = np.array(type_indices)[np.argsort([np.nanargmax((toRF_saccade_matrix[0][type_idx])) for type_idx in type_indices])][::-1]
        toRF_sorted_indices.append(sorted_type_indices_toRF)
        sorted_type_indices_awayRF = np.array(type_indices)[np.argsort([np.nanargmax((awayRF_saccade_matrix[0][type_idx])) for type_idx in type_indices])][::-1]
        awayRF_sorted_indices.append(sorted_type_indices_awayRF)


    # plot heatmaps
    sns.heatmap(np.array(toRF_saccade_matrix[coh])[np.concatenate(toRF_sorted_indices)],
                cmap='coolwarm', ax=axs[0, coh_index], vmin=-1, vmax=1, cbar=False)
    sns.heatmap(np.array(awayRF_saccade_matrix[coh])[np.concatenate(toRF_sorted_indices)],
                cmap='coolwarm', ax=axs[1, coh_index], vmin=-1, vmax=1, cbar=False)

    # styling
    axs[0, coh_index].set_title(f"toRF ({coh}%)")
    axs[1, coh_index].set_title(f"awayRF ({coh}%)")

    for row in [0, 1]:
        axs[row, coh_index].set_yticks([])
        axs[row, coh_index].axvline(300, color='red', linestyle='--')
        axs[row, coh_index].set_ylabel("Peak-Latency Sorted Neuron", fontsize=15)

    axs[0, coh_index].set_xticks([])
    axs[1, coh_index].set_xticks(np.arange(0, 310, 100))
    axs[1, coh_index].set_xticklabels(np.arange(-300, 10, 100))
    axs[1, coh_index].set_xlabel("Time (ms) relative to saccade onset")

for i in [0,1]:  
    start_num = 0
    for type_index, type in enumerate(neuron_types):
        n_neurons = len(toRF_sorted_indices[type_index])
        start_num += n_neurons
        for coh_index, coh in enumerate(coh_levels):
            axs[i, coh_index].axhline(start_num, color='black', linestyle='--', linewidth=1)
        # add text labels for each neuron type
        axs[i, 0].text(-5, start_num - n_neurons/2, type.replace('_',' ').title(), va='center', fontsize=10)
    
# Single colorbar
cbar_ax = fig.add_axes([1.01, 0.2, 0.01, 0.6])  # [left, bottom, width, height]
sm = plt.cm.ScalarMappable(cmap='coolwarm', norm=plt.Normalize(vmin=-1, vmax=1))
sm.set_array([])
fig.colorbar(sm, cax=cbar_ax)

plt.suptitle("Saccade-aligned population response (Z-scored)", fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


### Reaction time distribution

In [ ]:
coh_levels = [0,6,20,50]

toRF_rt = {coh:[] for coh in coh_levels}
awayRF_rt = {coh:[] for coh in coh_levels}

for neuron_id in neuron_metadata.neuron_id:
    
    session_id = neuron_metadata.session_id[neuron_metadata.neuron_id == neuron_id].values[0]
    trial_info = pd.read_csv(Path(compiled_dir, session_id, f"{session_id}_trial.csv"), index_col=None)
    trial_info = filter_trials(trial_info)
    
    trial_numbers = ephys_neuron_wise["cue"][neuron_id]['trial_number']

    toRF_trials = trial_info[trial_info['choice'] == 1]
    awayRF_trials = trial_info[trial_info['choice'] == 0]
    for coh in coh_levels:
        coh_toRF_trials = toRF_trials[toRF_trials['coherence'] == coh]
        coh_awayRF_trials = awayRF_trials[awayRF_trials['coherence'] == coh]
        
        toRF_rt[coh].append(coh_toRF_trials['reaction_time'])  
        awayRF_rt[coh].append(coh_awayRF_trials['reaction_time'])


In [ ]:
for coh in coh_levels:
    print(f"Coh {coh}, {np.nanmin(np.concatenate(toRF_rt[coh]))}, {np.nanmin(np.concatenate(awayRF_rt[coh]))}")

In [ ]:
fig, axs = plt.subplots(2, len(coh_levels), figsize=(6*len(coh_levels), 12))
for i, coh in enumerate(coh_levels):
    # Plotting for toRF
    axs[0, i].hist(np.concatenate(toRF_rt[coh]), bins=20,alpha=0.7)
    axs[0, i].set_title(f"toRF - Coh: {coh}")
    axs[0, i].set_ylabel("Frequency")

    # Plotting for awayRF
    axs[1, i].hist(np.concatenate(awayRF_rt[coh]), bins=20, alpha=0.7)
    axs[1, i].set_title(f"AwayRF - Coh: {coh}")
    axs[1, i].set_ylabel("Frequency")
plt.suptitle("Reaction Time Distribution by Coherence Level", fontsize=14)
plt.tight_layout()